# Goal Commitment in Humans and Machines

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import helpers as h
import pandas as pd
import numpy as np
from settings import data_dir as llm_data_dir

# Human data replication

Replication of Figures 2b and 2c from Holton et al. 2024 (Nat. Hum. Behav.).

- **Fig 2b**: Probability of goal abandonment as a function of the tree-search value of abandonment.
- **Fig 2c**: Persistence biases increase with goal progress (proportion of net completed).

In [ ]:
HUMAN_PATH = Path("data/human/learn_df.csv")
df = pd.read_csv(HUMAN_PATH)
plotter = h.Plotter(df, main_color="k", marker="X")
df.head()

In [ ]:
fig, ax, summary_2b = plotter.plot_fig2b()
fig.suptitle("Fig 2b", fontsize=10, y=1.02)
fig.tight_layout()

In [ ]:
fig, ax, summary_2c = plotter.plot_fig2c()
fig.suptitle("Fig 2c", fontsize=10, y=1.02)
fig.tight_layout()

# LLM data

In [ ]:
from_scratch = False
models = ["gemini-2.5-pro", "gpt-5", "claude-sonnet-4.5", "qwen3-32b", "centaur"]

model_colors_dict = {
    "human": "k",
    "gpt-5": "#12a37f", 
    "gemini-2.5-pro": "#F4B400", 
    "centaur": "#691a5e", 
    "claude-sonnet-4.5": "#da7756",
    "qwen3-32b": "#5a52d9",
}

model_markers_dict = {
    "human": "X",
    "gpt-5": "*", 
    "gemini-2.5-pro": "D", 
    "centaur": "h", 
    "claude-sonnet-4.5": "^",
    "qwen3-32b": "P",
}

model_names_dict = {
    "human": "Humans",
    "gpt-5": "GPT-5", 
    "gemini-2.5-pro": "Gemini 2.5 Pro", 
    "centaur": "Centaur", 
    "claude-sonnet-4.5": "Claude Sonnet 4.5",
    "qwen3-32b": "Qwen3 32B",
}

In [ ]:
plotters = {"human": plotter}

for model in models:
    family = h.get_family(model)
    if from_scratch:
        # Load raw LLM data (read-only source)
        model_df = pd.read_csv(f"{llm_data_dir}/{family}/{model}.csv")
        # Clean: drop participants with NaN choices, renumber IDs
        learn_df = h.prepare_learn_df(model_df)
        # Compute tree-search sampler values
        learn_df = h.compute_sampler_switch_value(learn_df)
        # Save learn_df.csv under holton/data/<family>/<model>/
        out_dir = Path("data") / family / model
        out_dir.mkdir(parents=True, exist_ok=True)
        learn_df.to_csv(out_dir / "learn_df.csv", index=False)
    else:
        # Load cleaned learn_df
        learn_df = pd.read_csv(f"data/{family}/{model}/learn_df.csv")

    # Create plotter
    plotters[model] = h.Plotter(
        learn_df,
        main_color=model_colors_dict.get(model, "#6a3d9a"),
        marker=model_markers_dict.get(model, "o"),
    )

    n_subj = learn_df["participant"].nunique()
    print(f"{model}: {n_subj} participants, {len(learn_df)} trials")

# Compare

In [ ]:
figures_dir = "results/figures/"
fmt = ".png"

In [ ]:
plotters.keys()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

#  Per-metric styling/options 
# Supported keys:
#   title, ylabel           (required)
#   ylim                    (optional, tuple) zooms the main y-axis
#   hline                   (optional, float) dashed reference line
#   inset_full_range        (optional, bool) add top-left inset with full range
#   broken_axis_inf         (optional, bool) stack a small inf band on top
metric_configs = {
    "nets_filled": {
        "title": "How many goals\nare completed?",
        "ylabel": "Total goals completed",
        "exclude": []
    },
    "persistence_bias": {
        "title": "How strong is\ngoal persistence?",
        "ylabel": "Indifference point",
        "hline": 0,
        "broken_axis_inf": True,
        "exclude": []
    },
    "best_alternative_chosen": {
        "title": "How often is the best alternative\nchosen when switching?",
        "ylabel": "P(best alternative chosen | switch)",
        "ylim": (0.5, 1.02),
        "exclude": ["claude-sonnet-4.5"]
    },
}

#  Pick which metrics to plot and in what order 
metrics_to_plot = [
    "persistence_bias",
    "nets_filled",
    "best_alternative_chosen",
]

N_COLS = 3
INCLUDE_LEGEND = True
LEGEND_I = 0
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (0.02, 0.95)

#  Helpers 
def _collect_finite_data(metric, exclude=[]):
    data = {}
    for model, p in plotters.items():
        if model in exclude:
            continue
        vals = p.metrics[metric]
        fin = vals[np.isfinite(vals)]
        if len(fin) > 0:
            data[model] = np.sort(fin)
    return data

def _collect_inf_counts(metric, exclude=[]):
    counts = {}
    for model, p in plotters.items():
        if model in exclude:
            continue
        vals = p.metrics[metric]
        n_inf = np.sum(np.isinf(vals))
        if n_inf > 0:
            counts[model] = (n_inf, len(vals))
    return counts

def _plot_cdf(ax, data_dict, title, ylabel, label=True):
    ax.set_title(title, fontsize=16)
    for model, vals in data_dict.items():
        p = plotters[model]
        x = np.arange(1, len(vals) + 1) / len(vals)
        ax.plot(x, vals, color=p.main_color,
                label=model_names_dict[model] if label else "_nolegend_",
                mec=p.main_color, mfc="white", markersize=6, marker=p.marker)
    ax.set_xlabel("Normalized subject number", fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)

def _add_inset_full_range(ax, data_dict, zoom_ylim):
    all_vals = np.concatenate(list(data_dict.values()))
    margin = 0.05 * (all_vals.max() - all_vals.min())
    inset = ax.inset_axes([0.08, 0.65, 0.38, 0.33])
    _plot_cdf(inset, data_dict, "", "", label=False)
    inset.set_ylim(all_vals.min() - margin, all_vals.max() + margin)
    inset.set_xlabel("")
    inset.set_ylabel("")
    inset.tick_params(labelsize=8)
    if zoom_ylim is not None:
        inset.axhspan(*zoom_ylim, color="lightgrey", alpha=0.4, zorder=0)

def _plot_broken_axis_inf(subplot_spec, metric, config):
    """Plot a metric with a broken y-axis for infinity values."""
    data = _collect_finite_data(metric, exclude=config.get("exclude", []))
    inf_counts = _collect_inf_counts(metric, exclude=config.get("exclude", []))
    has_inf = len(inf_counts) > 0

    if has_inf:
        inner = gridspec.GridSpecFromSubplotSpec(
            2, 1, subplot_spec=subplot_spec, height_ratios=[1, 6], hspace=0.08)
        ax_top = fig.add_subplot(inner[0])
        ax = fig.add_subplot(inner[1])
    else:
        ax = fig.add_subplot(subplot_spec)
        ax_top = None

    _plot_cdf(ax, data, "" if has_inf else config["title"], config["ylabel"])
    if "hline" in config:
        ax.axhline(config["hline"], color="k", linewidth=0.75,
                   linestyle="--", label="_nolegend_")
    if data:
        all_fin = np.concatenate(list(data.values()))
        margin = 0.15 * (all_fin.max() - all_fin.min())
        ax.set_ylim(all_fin.min() - margin, all_fin.max() + margin)

    if ax_top is not None:
        for model, (n_inf, n_total) in inf_counts.items():
            p = plotters[model]
            x = np.arange(1, n_inf + 1) / n_total
            y = np.full(n_inf, 1.0)
            label = model_names_dict[model] if model not in data else "_nolegend_"
            ax_top.plot(x, y, color=p.main_color, label=label,
                        mec=p.main_color, mfc="white", markersize=6, marker=p.marker)
        ax_top.set_ylim(0, 2)
        ax_top.set_yticks([1.0])
        ax_top.set_yticklabels(["$\\infty$"], fontsize=20)
        ax_top.tick_params(axis="x", bottom=False, labelbottom=False)
        ax_top.spines["bottom"].set_visible(False)
        ax.spines["top"].set_visible(False)
        d = 0.015
        kw = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1)
        ax_top.plot((-d, +d), (-d*6, +d*6), **kw)
        ax_top.plot((1 - d, 1 + d), (-d*6, +d*6), **kw)
        kw["transform"] = ax.transAxes
        ax.plot((-d, +d), (1 - d, 1 + d), **kw)
        ax.plot((1 - d, 1 + d), (1 - d, 1 + d), **kw)
        ax_top.set_title(config["title"], fontsize=16)

    return [a for a in [ax, ax_top] if a is not None]

def _plot_standard(subplot_spec, metric, config):
    data = _collect_finite_data(metric, exclude=config.get("exclude", []))
    ax = fig.add_subplot(subplot_spec)
    _plot_cdf(ax, data, config["title"], config["ylabel"])
    if "ylim" in config:
        ax.set_ylim(*config["ylim"])
    if "hline" in config:
        ax.axhline(config["hline"], color="k", linewidth=0.75,
                   linestyle="--", label="_nolegend_")
    if config.get("inset_full_range"):
        _add_inset_full_range(ax, data, config.get("ylim"))
    return [ax]

#  Figure layout 
n_cells = len(metrics_to_plot) + (1 if INCLUDE_LEGEND else 0)
n_rows = (n_cells + N_COLS - 1) // N_COLS

fig = plt.figure(figsize=(N_COLS * 5.3, n_rows * 5))
outer = gridspec.GridSpec(n_rows, N_COLS, wspace=0.35, hspace=0.4)

axes_for_legend = []
for i, metric in enumerate(metrics_to_plot):
    r, c = i // N_COLS, i % N_COLS
    config = metric_configs[metric]
    if config.get("broken_axis_inf"):
        axes_for_legend += _plot_broken_axis_inf(outer[r, c], metric, config)
    else:
        axes_for_legend += _plot_standard(outer[r, c], metric, config)

# Legend in the next empty cell
if INCLUDE_LEGEND:
    i_leg = LEGEND_I
    if i_leg < n_rows * N_COLS:
        r, c = i_leg // N_COLS, i_leg % N_COLS
        ax_leg = fig.add_subplot(outer[r, c])
        ax_leg.axis("off")
        handles, labels, seen = [], [], set()
        for ax in axes_for_legend:
            for hndl, lbl in zip(*ax.get_legend_handles_labels()):
                if lbl not in seen and lbl != "_nolegend_":
                    handles.append(hndl)
                    labels.append(lbl)
                    seen.add(lbl)
        ax_leg.legend(handles, labels, fontsize=13, loc=LEGEND_LOC, bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR, frameon=False)

fig.tight_layout()
fig.savefig(f"{figures_dir}goal_commitment_metrics{fmt}", dpi=300, bbox_inches="tight")

In [ ]:
metrics_to_test = {
    "persistence_bias": "Persistence bias (IP)",
    "nets_filled": "Nets filled",
    "best_alternative_chosen": "Best alternative chosen (on tempted switches)",
}

for metric, name in metrics_to_test.items():
    print("=" * 60)
    print(f"{name} vs. humans")
    print("=" * 60)
    human_vals = plotters["human"].metrics[metric]
    human_vals = human_vals[np.isfinite(human_vals)]
    exclude = metric_configs.get(metric, {}).get("exclude", [])

    for model in plotters:
        if model == "human":
            continue
        if model in exclude:
            print(f"{model}: (excluded — not enough data to assess)")
            continue
        model_vals = plotters[model].metrics[metric]
        model_vals = model_vals[np.isfinite(model_vals)]

        if len(model_vals) == 0:
            print(f"{model}:\n    No measurable values (all NaN/inf)")
            continue

        direction = "higher" if np.mean(model_vals) > np.mean(human_vals) else "lower"
        print(f"{model} (average {direction} than humans):")
        print("    ", end="")
        h.print_kstest_results(model_vals, human_vals)
        print("    ", end="")
        h.print_mannwhitney_results(model_vals, human_vals)

In [ ]:
from IPython.display import display, HTML

prefixes = {
    "persistence_bias": "IP",
    "nets_filled": "N. goals",
    "best_alternative_chosen": "P(best)",
}

# Read per-metric exclusions from metric_configs (defined in the plot cell)
exclude_per_metric = {
    m: metric_configs.get(m, {}).get("exclude", []) for m in prefixes
}

desc_df, stats_df = h.build_comparison_tables(
    plotters, models, prefixes, model_names_dict,
    exclude_per_metric=exclude_per_metric,
)

display(HTML("<h4>Descriptive statistics</h4>"))
display(desc_df.style.set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
    {"selector": "td", "props": [("text-align", "center")]},
]))
display(HTML("<h4>Comparison to humans (* p&lt;.05, ** p&lt;.01, *** p&lt;.001)</h4>"))
display(stats_df.style.set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
]))